# Hyper-parameter Tuning

### Learning Objectives

In this notebook we learn to:

1. Given Linear Regression or Decision Tree model, distinguish between it's *parameters* and *hyper-parameters*
2. Recognize the 5 elements of hyper-parameter search in scikit-learn
3. Explain the process of hyper-parameter search; describing the role of each of the 5 elements
4. Explain the difference between Grid Search and either Random Search or Bayesian Search
5. Recognize the more efficient, model-specific cross-validations


**Hyper-parameters** are parameters that are not directly learnt within estimators . In scikit-learn they are passed as arguments to the constructor of the estimator classes. Examples include:

- for `LinearRegression`: the degree of the polynomial
- for `LogisticRegression`: the regularization strength `C`
- for Decision Trees: `max_depth`, `min_samples_leaf`
- for `SVM`: `C`, `kernel` and `gamma`

It is possible and **recommended to search the hyper-parameter space for the best** [**cross validation**](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation) **score**.

Any parameter provided when constructing an estimator may be optimized in this manner. Specifically, to find the names and current values for all parameters for a given estimator, use:

```python
estimator.get_params()
```

The five elements of HP search are:

1. an *estimator*
2. a parameter space
3. a cross-validation scheme
4. a search method (tuning algorithms)
5. a [score function](https://scikit-learn.org/stable/modules/grid_search.html#gridsearch-scoring) (examplee: `accuracy_score` in classification)

## Hyper-parameter tuning methods

Two generic approaches to parameter search are provided in scikit-learn: for given values:

1. [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV "sklearn.model_selection.GridSearchCV") exhaustively considers all parameter combinations
2. [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV "sklearn.model_selection.RandomizedSearchCV") can sample a given number of candidates from a parameter space with a specified distribution.

Example: decision trees have `n_estimators` and `max_depth`:

![](../assets/gridsearch_vs_randomsearch.png)

For more hyper-parameters, imagine more dimensions (not just 2D grid).


## Parameter Tuning Workflow

Here is a flowchart of typical cross validation workflow in model training. The best parameters can be determined by the abovementioned techniques.

```{mermaid}
graph TD
    %% Define color styling for nodes
    classDef purpleNode fill:#ccccff,stroke:#000,stroke-width:1px,color:#000;

    %% Define the nodes
    Parameters(param_grid):::purpleNode
    Dataset(data):::purpleNode
    Crossvalidation(GridSearchCV):::purpleNode
    Trainingdata(train_data):::purpleNode
    Testdata(test_data):::purpleNode
    Bestparameters(search.best_params_):::purpleNode
    Retrainedmodel(search.best_estimator_):::purpleNode
    Finalevaluation(".score()"):::purpleNode

    %% Define the connections
    Dataset --> Trainingdata
    Dataset --> Testdata
    Parameters --> Crossvalidation
    Trainingdata --> Crossvalidation
    Crossvalidation --> Bestparameters
    Trainingdata --> Retrainedmodel
    Bestparameters --> Retrainedmodel
    Retrainedmodel --> Finalevaluation
    Testdata --> Finalevaluation
```

- The data is split intro training and testing sets
- The search space `param_grid` is specified

Hyper-parameter search algorithm `GridSearchCV` iterate on the train-validation splits; for each model doing training and scoring.

The resulting `search.best_params` are used to initialize a new model and fit it again on the entire train set.

Finally, the trained model is evaluated on the test set.

In the following example, we randomly search over the parameter space of a random forest with a [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV) object.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from scipy.stats import randint

In [ ]:
# create a synthetic dataset
X, y = make_regression(
    n_samples=20640,
    n_features=8,
    noise=0.1,
    random_state=0
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

In [ ]:
# define the parameter space that will be searched over
param_distributions = {
    'n_estimators': randint(1, 5),
    'max_depth': randint(5, 10)
}
# now create a searchCV object and fit it to the data
search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=0),
    n_iter=5,
    param_distributions=param_distributions,
    random_state=0
)

search.fit(X_train, y_train)

![Figure: Cross Validation](../assets/inner_cross_validation.png)

In [ ]:
model = search.best_estimator_
model

array([ 284.79557583,   52.55187449,   36.64321571, ..., -136.91520557,
        126.28826424,  116.76904048], shape=(5160,))

In [ ]:
model.score(X_test, y_test)

## Use with Pipelines

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV "sklearn.model_selection.GridSearchCV") and [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV "sklearn.model_selection.RandomizedSearchCV") allow searching over parameters of composite or nested estimators such as [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline "sklearn.pipeline.Pipeline"), [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html#sklearn.compose.ColumnTransformer "sklearn.compose.ColumnTransformer"), [`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html#sklearn.ensemble.VotingClassifier "sklearn.ensemble.VotingClassifier") or [`CalibratedClassifierCV`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html#sklearn.calibration.CalibratedClassifierCV "sklearn.calibration.CalibratedClassifierCV") using a dedicated `<estimator>__<parameter>` syntax:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import make_moons


In [ ]:
# 1. Prepare Data
X, y = make_moons(n_samples=500, noise=0.3, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)

In [ ]:
# 2. Define the Pipeline
# We treat the scaler and the classifier as a single "mega-model."
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

In [ ]:

# 3. Define the Search Grid
# Syntax: <step_name>__<parameter_name>
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5, None]
}

search = GridSearchCV(pipe, param_grid, cv=5)

In [ ]:
# 4. Run the Grid Search
# This automatically scales the data and trains the model for every combination.
search.fit(X_train, y_train)

print(f"Best settings found: {search.best_params_}")
print(f"Accuracy of best model: {search.best_score_:.2%}")

Best settings found: {'model__max_depth': 5, 'model__n_estimators': 100}
Accuracy of best model: 89.60%


In [ ]:
best_model = search.best_estimator_
best_model

In [ ]:
best_model.score(X_test, y_test)

Refer to [Pipeline: chaining estimators](https://scikit-learn.org/stable/modules/compose.html#pipeline) for more details.

## Variants

### A. Halving

Both `GridSearchCV` and `RandomSearchCV` have successive halving counterparts:

1. [`HalvingGridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingGridSearchCV.html#sklearn.model_selection.HalvingGridSearchCV "sklearn.model_selection.HalvingGridSearchCV")
2. [`HalvingRandomSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html#sklearn.model_selection.HalvingRandomSearchCV "sklearn.model_selection.HalvingRandomSearchCV")

.. which can be much faster at finding a good parameter combination.

### B. Model-specific cross-validation

Better yet, use a [Model specific cross-validation](https://scikit-learn.org/stable/modules/grid_search.html#model-specific-cross-validation) for example:
  - Regressor model: `RidgeCV`
  - Classifier model: `LogisticRegressionCV`

### C. Bayesian Optimization

If none was found, you may want to use [**Bayesian Optimization**](https://scikit-optimize.github.io/stable/modules/generated/skopt.BayesSearchCV.html) which treats the search as an optimization problem itself to find the best values with as few trials as possible. On some datasets, it leads to the same score with: **7x fewer iterations and 5x less time**.